# Lecture 1 — From LLM to Agent: The Simplest Possible Loop

**Central question:** What is the *minimum* structure that turns an LLM call into an agent?

We will build up from scratch — no frameworks, just the raw OpenAI API.

---
## Setup

In [17]:
from typing import Any, Callable, List, cast
from openai import OpenAI
import sys, json
sys.path.insert(0, '..')
from shared.utils import get_openai_client, print_tool_call, print_messages

client: OpenAI = get_openai_client()
MODEL: str = "gpt-4o-mini"
print("Client ready.")

Client ready.


In [18]:
import importlib
import shared.utils
importlib.reload(shared.utils)

<module 'shared.utils' from 'c:\\git\\ai-agents\\lecture1_minimal_loop\\..\\shared\\utils.py'>

---
## Part 1 — A Bare API Call

The simplest possible interaction: send a message, receive a completion.

### Anatomy of the request
- **model** — which LLM to use
- **messages** — the conversation so far, as a list of `{role, content}` dicts
- **max_tokens** — budget for the response (tokens ≈ words)

Every API call is stateless. The model has no memory between calls — we send the *entire* conversation each time.

In [19]:
from openai.types.chat import ChatCompletion, ChatCompletionMessageParam

response: ChatCompletion = client.chat.completions.create(
    model = MODEL,
    messages = cast(List[ChatCompletionMessageParam], [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is the capital of France?"},
    ]),
    max_tokens=64,
)

print(response.choices[0].message.content)

The capital of France is Paris.


### Inspect the full response object

There is more here than just the text.

In [20]:
print("Finish reason:", response.choices[0].finish_reason)
if (usage := response.usage):
    print("Tokens used  — prompt:", usage.prompt_tokens,
          "| completion:", usage.completion_tokens,
          "| total:", usage.total_tokens)

Finish reason: stop
Tokens used  — prompt: 24 | completion: 7 | total: 31


`finish_reason` tells us *why* the model stopped:
- `stop` — natural end of response  
- `length` — hit `max_tokens`  
- `tool_calls` — model wants to call a tool (we will see this soon)

**Discussion:** Why does every API call send the full conversation history? What does this tell us about where "memory" lives?

---
## Part 2 — What's Missing?

The model can answer questions about things it knows from training.  
But what if we ask something it *cannot* know?

In [21]:
response: ChatCompletion = client.chat.completions.create(
    model = MODEL,
    messages = cast(List[ChatCompletionMessageParam], [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What time is it right now?"},
    ]),
    max_tokens=64,
)

print(response.choices[0].message.content)

I'm sorry, but I cannot provide real-time information such as the current time. You can check the time on your device or a clock nearby.


The model can only generate text. It cannot *do* anything: no clock, no internet, no file system.

To give it capabilities, we need to give it **tools**.

---
## Part 3 — Adding a Tool

We define a tool as a **JSON schema**: name, description, and parameter types.  
The model reads this schema and decides when to call the tool.

We write the actual Python function. The model never runs code — it only emits a structured request.

In [22]:
import datetime
from openai.types.chat import ChatCompletionToolParam

# The real Python function
def get_current_time() -> str:
    """Return the current UTC time as a string."""
    current_time = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"Tool call: {current_time}")
    return current_time

# The tool schema we send to the model for explaining available tools
tools: List[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns the current UTC date and time.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    }
]

print("Tool defined. Schema sent to model:")
print(json.dumps(tools[0]["function"], indent=2))

Tool defined. Schema sent to model:
{
  "name": "get_current_time",
  "description": "Returns the current UTC date and time.",
  "parameters": {
    "type": "object",
    "properties": {},
    "required": []
  }
}


### Call the API with the tool available

In [28]:
from openai.types.chat import ChatCompletionMessage

# Prepare the conversation
messages: List[ChatCompletionMessageParam] = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What time is it right now?"},
]

# Send to the model for a response
response: ChatCompletion = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

# Extract response message
msg: ChatCompletionMessage = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason} content: {msg.content} tokens: {response.usage.completion_tokens}")
print()

# Since we are expecting a tool callback, show it (using pretty-printed helper from utils)
if msg.tool_calls:
    print("Model wants to call a tool:")
    for tc in msg.tool_calls:
        print_tool_call(tc)
    print()
    print("Raw tool_call JSON (this is THE interface):")
    print(json.dumps(json.loads(msg.tool_calls[0].function.arguments), indent=2))

Finish reason: tool_calls content: None tokens: 11

Model wants to call a tool:
  Function tool: get_current_time Args: {}

Raw tool_call JSON (this is THE interface):
{}


**Stop here.** Look at that JSON string on screen.

> *This string is the interface.*

In classical software, a function call is enforced by the compiler — wrong types won't compile.  
Here, the "call" is a piece of natural language structured as JSON. The model *generates* it. Nothing enforces it.

**Discussion:** What could go wrong with this interface? What happens if the model hallucinates a parameter name?

---
## Part 4 — Closing the Loop

The model asked for a tool. Now we:
1. Parse the request
2. Run our Python function
3. Return the result as a new message
4. Let the model continue

This is **one full turn** of the agent loop.

In [29]:
# from openai.types.chat.chat_completion_message_tool_call import ChatCompletionMessageToolCallUnion

# We already have: messages (original), response (model's first reply)
# tool_call: ChatCompletionMessageToolCallUnion

from shared.utils import to_param
if (tool_calls := response.choices[0].message.tool_calls):
    tool_call = tool_calls[0]

    # Step 1: append the model's assistant message (it contains the tool request)
    messages.append(to_param(response.choices[0].message))

    # Step 2: call our Python function
    tool_result: str = get_current_time()
    print("Tool returned:", tool_result)

    # Step 3: append the tool result as a "tool" message
    messages.append(cast(ChatCompletionMessageParam, {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": tool_result,
    }))

    # Step 4: call the model again — now it has the result in context
    response2: ChatCompletion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
    )

    print()
    print(f"Model's final reply: {response2.choices[0].message.content} tokens: {response2.usage.completion_tokens}")
    print()

C:\Users\Eugene\AppData\Local\Temp\ipykernel_7992\912769658.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_time = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


Tool call: 2026-03-09 20:25:55 UTC
Tool returned: 2026-03-09 20:25:55 UTC

Model's final reply: The current time is 20:25:55 UTC on March 9, 2026. tokens: 21



### Inspect the full message history

In [30]:
from openai.types.chat import ChatCompletionMessageToolCallUnion

messages.append({"role": "assistant", "content": response2.choices[0].message.content})

print("Full conversation (what the model 'sees' on each call):")
print()
for i, m in enumerate(messages):
    role: str = m["role"].upper()
    content: str = m["content"] or "TOOL CALL: " + str(cast(List[ChatCompletionMessageToolCallUnion], m["tool_calls"])[0].function)
    if len(content) > 200:
        content = content[:200] + "..."

    print(f"{i}. [{role}] {content}")

Full conversation (what the model 'sees' on each call):

0. [SYSTEM] You are a helpful assistant.
1. [USER] What time is it right now?
2. [ASSISTANT] TOOL CALL: Function(arguments='{}', name='get_current_time')
3. [TOOL] 2026-03-09 20:25:55 UTC
4. [ASSISTANT] The current time is 20:25:55 UTC on March 9, 2026.


---
## Part 5 — The Agent Loop

One turn was enough for this task. But what if the model needs to call *multiple* tools in sequence?  
We need to loop until `finish_reason == 'stop'`.

Let's build a general loop — and give the model a slightly more interesting task.

In [ ]:
# A second tool: calculator
def calculate(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""
    try:
        # Restrict to safe operations only
        allowed: set[str] = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expression):
            return "Error: unsupported characters in expression."
        result: Any = eval(expression, {"__builtins__": {}}, {})
        print(f"Calculate tool: {str(result)}")
        return str(result)
    except Exception as e:
        return f"Error: {e}"

tools2: List[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns the current UTC date and time.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a simple arithmetic expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Arithmetic expression, e.g. '3 * (4 + 2)'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
]

# Dispatch table: maps tool name → Python function
TOOL_FUNCTIONS: dict[str, Callable[[dict[str, Any]], str]] = {
    "get_current_time": lambda args: get_current_time(),
    "calculate": lambda args: calculate(args["expression"]),
}

print("Tools registered:", list(TOOL_FUNCTIONS.keys()))

In [ ]:
def run_agent(
    user_message: str,
    tools: List[ChatCompletionToolParam],
    max_turns: int = 10,
) -> str:
    """
    Minimal agent loop.
    Observe (tool result) → Think (model) → Act (tool call) — repeat until done.
    """
    # list[Any] because we mix ChatCompletionMessageParam dicts and ChatCompletionMessage objects
    agent_messages: list[Any] = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user",   "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"--- Turn {turn + 1} ---")

        response: ChatCompletion = client.chat.completions.create(
            model=MODEL,
            messages=agent_messages,  # type: ignore[arg-type]
            tools=tools,
        )

        msg: ChatCompletionMessage = response.choices[0].message
        finish: str | None = response.choices[0].finish_reason
        print(f"Finish reason: {finish}")

        # Always append the assistant's message (may contain tool_calls)
        agent_messages.append(msg)

        if finish == "stop":
            # Model is done — return its final text
            return msg.content or ""

        if finish == "tool_calls" and msg.tool_calls:
            # Execute each requested tool
            for tc in msg.tool_calls:
                print_tool_call(tc)
                args: dict[str, Any] = json.loads(tc.function.arguments)
                result: str = TOOL_FUNCTIONS[tc.function.name](args)
                print(f"  Result: {result}")
                agent_messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
            # Loop — let the model continue
            continue

        # Unexpected finish reason
        print(f"Unexpected finish reason: {finish}")
        break

    return "[max_turns reached]"

print("Agent loop defined.")

### Run it

In [ ]:
answer: str = run_agent(
    "What time is it, and what is 1234 * 5678?",
    tools=tools2,
)

print()
print("Final answer:")
print(answer)

### Observe → Think → Act

Each turn follows the same pattern:

```
Observe  — tool result arrives in the message list
Think    — model reads full context, generates next output
Act      — output is either a tool call or a final answer
```

This loop **is** the agent. Everything else — memory, planning, multi-agent systems — is elaboration on this core.

---
## Deliberately Break It

What happens if we ask for a tool that doesn't exist?

In [ ]:
# Register tools but leave the dispatch table incomplete
broken_tool: List[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"],
            },
        },
    }
]

BROKEN_FUNCTIONS: dict[str, Callable[[dict[str, Any]], str]] = {}  # deliberately empty

broken_messages: List[ChatCompletionMessageParam] = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What's the weather in Paris?"},
]

response: ChatCompletion = client.chat.completions.create(
    model=MODEL,
    messages=broken_messages,
    tools=broken_tool,
)

msg: ChatCompletionMessage = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)
if msg.tool_calls:
    tc = msg.tool_calls[0]
    print_tool_call(tc)
    try:
        result: str = BROKEN_FUNCTIONS[tc.function.name]({})
    except KeyError:
        print(f"  KeyError: '{tc.function.name}' not in dispatch table.")
        print("  The model asked for a tool we never implemented.")

**Key point:** The model faithfully follows its description and generates a valid-looking tool call. The failure is silent — it happens *after* the model, in our Python code.

The natural-language interface has no enforcement mechanism. This is both the power and the fragility of agent systems.

---
## Part 6 — The Higher-Level Alternative: Auto-Schema from Python Functions

Everything we did in Parts 3–5 by hand — writing the JSON schema, building the dispatch table — is pure boilerplate.  
Real frameworks (LangChain, OpenAI Agents SDK, etc.) eliminate it with a **decorator** that reads your function's type hints and docstring.

Let's build the minimal version of that abstraction ourselves, so you can see exactly what it does.

In [ ]:
import inspect
from typing import get_type_hints

# --- Tiny tool registry ---
_tool_registry: dict[str, Callable[[dict[str, Any]], str]] = {}
_tool_schemas: list[ChatCompletionToolParam] = []

# Map Python built-in types → JSON Schema type strings
_PY_TO_JSON: dict[type, str] = {str: "string", int: "integer", float: "number", bool: "boolean"}

def tool(fn: Callable[..., str]) -> Callable[..., str]:
    """
    Decorator: reads type hints + docstring → builds JSON schema → registers for dispatch.
    This is what LangChain's @tool does internally.
    """
    hints: dict[str, Any] = get_type_hints(fn)
    hints.pop("return", None)   # return type is not part of the schema
    sig = inspect.signature(fn)

    # Build parameters dict from type hints and default values
    properties: dict[str, Any] = {}
    required: list[str] = []
    for name, py_type in hints.items():
        param = sig.parameters.get(name)
        properties[name] = {
            "type": _PY_TO_JSON.get(py_type, "string"),
            "description": f"{name} ({py_type.__name__})",
        }
        # No default → required
        if param and param.default is inspect.Parameter.empty:
            required.append(name)

    schema: ChatCompletionToolParam = {
        "type": "function",
        "function": {
            "name": fn.__name__,
            "description": (fn.__doc__ or "").strip(),
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }

    # Register schema and dispatch entry
    _tool_schemas.append(schema)
    _tool_registry[fn.__name__] = lambda args: fn(**args)

    return fn   # return the original function unchanged


# --- Define tools with the decorator — no manual JSON schema ---

@tool
def get_current_time2() -> str:
    """Returns the current UTC date and time."""
    import datetime
    return datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d %H:%M:%S UTC")

@tool
def calculate2(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the result."""
    allowed: set[str] = set("0123456789+-*/()., ")
    if not all(c in allowed for c in expression):
        return "Error: unsupported characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


print("Registered tools:", list(_tool_registry.keys()))
print()
print("Auto-generated schema for calculate2:")
print(json.dumps(_tool_schemas[1]["function"], indent=2))

Notice: the schema above was generated entirely from the function signature and docstring.  
We wrote *zero* JSON by hand.

Now run the same agent loop — the only difference is that `_tool_schemas` and `_tool_registry` are populated by the decorator rather than by us.

In [ ]:
def run_agent_auto(user_message: str, max_turns: int = 10) -> str:
    """
    Same loop as run_agent(), but uses the decorator-populated registry.
    No tool list or dispatch table passed in manually.
    """
    messages: list[Any] = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user",   "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"--- Turn {turn + 1} ---")
        response: ChatCompletion = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=_tool_schemas,        # auto-generated schemas
        )
        msg: ChatCompletionMessage = response.choices[0].message
        finish: str | None = response.choices[0].finish_reason
        print(f"Finish reason: {finish}")
        messages.append(msg)

        if finish == "stop":
            return msg.content or ""

        if finish == "tool_calls" and msg.tool_calls:
            for tc in msg.tool_calls:
                print_tool_call(tc)
                args: dict[str, Any] = json.loads(tc.function.arguments)
                # Dispatch via registry — no if/elif chain needed
                result: str = _tool_registry[tc.function.name](args)
                print(f"  Result: {result}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            continue

        break

    return "[max_turns reached]"


answer2: str = run_agent_auto("What time is it, and what is 1234 * 5678?")
print()
print("Final answer:", answer2)

### What changed — and what did not

| | Manual (Parts 3–5) | Decorator (Part 6) |
|---|---|---|
| JSON schema | Written by hand | Generated from type hints + docstring |
| Dispatch table | `TOOL_FUNCTIONS` dict, maintained manually | `_tool_registry`, populated by `@tool` |
| Agent loop | Identical | Identical |
| OpenAI API calls | Identical | Identical |

**The loop is the same.** The API calls are the same. The model sees the same JSON on the wire.  
The decorator is purely a developer-experience improvement — it does not change the protocol at all.

> *When you use LangChain's `@tool`, LangGraph's `ToolNode`, or the OpenAI Agents SDK — this is exactly what they are doing behind the scenes.*  
> *The strings still flow. The loop still runs.*

---
## Summary

| Step | What we did | Key insight |
|------|-------------|-------------|
| 1 | Bare API call | Model is stateless; we send full history each time |
| 2 | Identified the gap | Model can't act on the world without tools |
| 3 | Added a tool | Interface = JSON schema in, JSON call out |
| 4 | Closed the loop | Tool result becomes the next message |
| 5 | Generalized to a loop | Observe → Think → Act — this is an agent |

> *In classical software, interfaces are enforced by type systems.*  
> *In agent systems, the interface between components is natural language — strings flowing between nodes.*

**This will be the recurring theme across all four lectures.**

---
### Discussion Prompts

1. What would happen if the tool returned structured data (a dict) instead of a string?
2. Where is the "decision" happening? Is the model really deciding?
3. What could go wrong in this loop? When might it not terminate?